In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv(path + "/Q3_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info

In [ ]:
# Task 4: Write your code here:
df.head()

In [ ]:
# Task 1: Write your code here:
print("Missing values before:\n", df.isnull().sum().sum())


for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

print("Missing values after:\n", df.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Duplicates found: {duplicates}")

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates removed.")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include=['object', 'category']).columns

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Categorical variables encoded.")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include=['object', 'category']).columns

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Categorical variables encoded.")

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
target_col = 'target'
if 'target' not in df.columns:
    target_col = df.columns[-1]

feature_cols = [c for c in df.columns if c != target_col]
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Target Value Counts:")
print(df[target_col].value_counts(normalize=True))

is_imbalanced = df[target_col].value_counts(normalize=True).min() < 0.2
print(f"Is data imbalanced? {'Yes' if is_imbalanced else 'No'}")

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=[target_col])
y = df[target_col]

In [ ]:
# Task 2,3,4,5: Write your code here:
!pip install catboost
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []
acc_scores = []

fold = 1
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1, verbose=0, random_seed=42)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    f1_scores.append(f1)
    acc_scores.append(acc)

    print(f"Fold {fold} - F1 Score: {f1:.4f} | Accuracy: {acc:.4f}")
    fold += 1

print(f"\nAverage F1 Score: {np.mean(f1_scores):.4f}")
print(f"Average Accuracy: {np.mean(acc_scores):.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

feature_importances = model.get_feature_importance()
feature_names = X.columns

fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
fi_df = fi_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=fi_df.head(10)) # Top 10 features
plt.title('Top 10 Feature Importances')
plt.show()



In [ ]:
# Task 2: Write your code here:
golden_feature = fi_df.iloc[0]['Feature']
print(f"THE GOLDEN FEATURE IS: {golden_feature}")

In [ ]:
# Task Bonus: Write your code here:
X_gold = X[[golden_feature]]

# NOw we Run StratifiedKFold loop
gold_f1_scores = []

print(f"Retraining with ONLY {golden_feature}...")

for train_index, test_index in skf.split(X_gold, y):
    X_train, X_test = X_gold.iloc[train_index], X_gold.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_gold = CatBoostClassifier(iterations=200, verbose=0, random_seed=42)
    model_gold.fit(X_train, y_train)
    y_pred = model_gold.predict(X_test)

    gold_f1_scores.append(f1_score(y_test, y_pred))

# 3. Print and compare
avg_gold_f1 = np.mean(gold_f1_scores)
print(f"\nAverage F1 Score (Golden_Feature_Only): {avg_gold_f1:.4f}")
print(f"Average F1 Score (All_Features): {np.mean(f1_scores):.4f}")

if avg_gold_f1 >= 0.9 * np.mean(f1_scores):
    print("Insight: The Golden Feature captures the vast majority of the predictive power!")
else:
    print("Insight: The Golden Feature is strong, but other features add significant value.")